### Conversation for an insurance-company customer support flow. 
- **It uses the same ConversationChain with buffer memory to keep track of the dialogue.**
---

- Simulated Dialogue

- The customer first asks about claim status.

- They supply their claim number.

- They follow up asking which documents are still pending.

- At last, they request a reminder of specific policy coverages.

- Final Summary Step
After your normal Q&A turns, you invoke the summary chain, producing a 
high-level recap of what the customer has asked, which you can log, display, or feed into a 
human-agent dashboard before closing the session.

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

# use this for direct cohere model access using cohere-api-key

# # Initialize the Cohere LLM
# YOUR_COHERE_API_KEY = ""

# from langchain.llms import Cohere
# llm = Cohere(cohere_api_key=YOUR_COHERE_API_KEY, temperature=0.7)

# OCI Generative AI service LLM Access

# Import LangChain components
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

# Initialize the Cohere language model
llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)

In [ ]:
# from langchain_cohere import ChatCohere
from langchain.chains import ConversationChain
from langchain.chains.conversation.memory import ConversationBufferMemory
import warnings
from IPython.display import Markdown, display


# ─────────────────────────────────────────────────────────────────────────────
# Setup
# ─────────────────────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore", category=DeprecationWarning)

def get_AIresponse(chain, query: str) -> str:
    """Run the conversation chain on a user query."""
    return chain.invoke(query)

# ─────────────────────────────────────────────────────────────────────────────
# Memory‐backed Conversation Chain
# ─────────────────────────────────────────────────────────────────────────────
conversation = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory()
)

# ─────────────────────────────────────────────────────────────────────────────
#  Insurance Dialog
# ─────────────────────────────────────────────────────────────────────────────
# 1) Customer greets the system
# print("--- Customer → AI ---")
display(Markdown( "**--- Customer → AI ---**"))

display(Markdown( "**Hello, I’d like to check on my claim status**"))
# Seed the conversation
_ = conversation.run("Hello, I’d like to check on my claim status.")

# 2) Customer provides a claim number
query = "My claim number is CLM-20250610-1234. What’s the current status?"

display(Markdown( "**--- Customer → AI ---**"))
display(Markdown( "**"+query+"**"))

output = get_AIresponse(conversation, query)

display(Markdown( "**--- AI → Customer ---**"))

print(output['response'])

# 3) Customer asks what documents are still required
query = "Great, thanks. Which documents do I still need to upload to complete this claim?"

display(Markdown( "**--- Customer → AI ---**"))

display(Markdown( "**"+query+"**"))
output = get_AIresponse(conversation, query)

display(Markdown( "**--- AI → Customer ---**"))
print(output['response'])

# 4) Customer requests a summary of their policy coverages
query = (
    "Also, can you remind me what my policy covers? "
    "Specifically for roadside assistance and rental car coverage."
)

display(Markdown( "**--- Customer → AI ---**"))
display(Markdown( "**"+query+"**"))

output = get_AIresponse(conversation, query)

display(Markdown( "**--- AI → Customer ---**"))
print(output['response'])

# ─────────────────────────────────────────────────────────────────────────────
# 📋  Summarize Conversation Before Ending Chat
# ─────────────────────────────────────────────────────────────────────────────
# Build a summarization chain that ingests the full chat history
from langchain.prompts import PromptTemplate
summary_prompt = PromptTemplate.from_template(
    "Below is the full conversation between the customer and an insurance support AI:\n\n"
    "{collected_chat_history}\n\n"
    "Please summarize the customer's requests and key details so far in 3–4 bullet points."
)
# summary_chain = LLMChain(llm=llm, prompt=summary_prompt)

summary_chain = summary_prompt | llm


# Pull the raw chat buffer from memory
chat_history = conversation.memory.buffer

# Generate and print the summary
summary = summary_chain.invoke({"collected_chat_history":chat_history})

display(Markdown( "**--- Conversation Summary (for internal use) ---**"))

display(Markdown(summary.content))




In [ ]:
# print(conversation.memory.buffer)   # pint the content in memory i.e. chat history
# conversation_buf.memory.clear()    # clear the history if needed